# Cambiar ropa por uniforme SENATI

Subes una foto de una persona y el cuaderno devuelve **la misma foto** (misma cara, misma pose, mismo fondo)
pero con el uniforme SENATI puesto: uniforme de hombre para los chicos y uniforme de mujer para las chicas.

Usa **CatVTON**, un modelo gratuito de prueba virtual de ropa que ya viene entrenado con miles de fotos.
Todo corre en Colab; tu PC no hace nada.

## Para el evento: corre solo los pasos 1, 2, 3, 4, 5 y 8

1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** y guardar.
2. Toca ▶ en los pasos **1, 2, 3, 4, 5 y 8**, uno por uno y en ese orden. Espera que cada uno termine.
3. El paso 2 reinicia la sesión al terminar (sale «La sesión se bloqueó»): es normal, sigue con el 3.
4. El paso 3 no pide nada: los uniformes se bajan solos de la página.
5. Cuando el paso 8 diga **«Motor encendido»**, deja esta pestaña abierta todo el evento.

Los pasos 6 y 7 **no se usan en el evento** (son para probar fotos sin la página).

## 1. Comprobar la GPU

In [ ]:
#@title 1. Comprobar la GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No hay GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno → GPU T4."
print("GPU lista:", torch.cuda.get_device_name(0))

## 2. Instalar el modelo (tarda unos 2 minutos)

Al terminar, la sesión se reinicia sola para cargar las librerías nuevas
(sale el aviso «La sesión se bloqueó»; es normal). Después sigue con el paso 3, o con el 4 si es para el evento.

In [ ]:
#@title 2. Instalar el modelo (tarda unos 2 minutos)
%cd /content
!rm -rf CatVTON && git clone -q --depth 1 https://github.com/Zheng-Chong/CatVTON.git
!pip install -q "diffusers>=0.30" fvcore==0.1.5.post20221221 omegaconf pycocotools av cloudpickle "rembg[cpu]" gradio
print("Instalación terminada. Reiniciando la sesión...")
# Reinicio para no mezclar la versión vieja de Pillow cargada en memoria con la nueva
import os
os.kill(os.getpid(), 9)

## 3. Subir las fotos

Elige el archivo `datos_para_colab.zip`. Dentro debe tener:
- carpeta `hombre/` con fotos de chicos
- carpeta `mujeres/` con fotos de chicas
- `uniforme_hombre.png` y `uniforme_mujer.png`

In [ ]:
#@title 3. Subir las fotos
SUBIR_ZIP = False  #@param {type:"boolean"}
import os, glob
from google.colab import files

if not SUBIR_ZIP:
    print("Para el evento no hace falta subir nada: sigue con el paso 4.")
else:
    subido = files.upload()
    nombre_zip = next(iter(subido))
    !rm -rf /content/datos && mkdir -p /content/datos
    !unzip -q -o "/content/{nombre_zip}" -d /content/datos
    for carpeta in ["hombre", "mujeres"]:
        print(carpeta, ":", len(glob.glob(f"/content/datos/{carpeta}/*")), "fotos")
    print("Uniformes:", sorted(os.path.basename(p) for p in glob.glob("/content/datos/uniforme_*")))

## 4. Cargar el modelo (tarda unos 3 minutos la primera vez)

In [ ]:
#@title 4. Cargar el modelo (tarda unos 3 minutos la primera vez)
import glob, os, sys
os.chdir("/content/CatVTON")
sys.path.insert(0, "/content/CatVTON")

import cv2
import numpy as np
import torch
from PIL import Image, ImageFilter, ImageOps
from huggingface_hub import snapshot_download
from diffusers.image_processor import VaeImageProcessor
from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline

ruta_modelo = snapshot_download("zhengchong/CatVTON", allow_patterns=["mix-48k-1024/*", "DensePose/*", "SCHP/*"])

# La T4 no tiene bfloat16, por eso se usa float16
pipeline = CatVTONPipeline(
    base_ckpt="booksforcharlie/stable-diffusion-inpainting",
    attn_ckpt=ruta_modelo,
    attn_ckpt_version="mix",
    weight_dtype=torch.float16,
    use_tf32=True,
    device="cuda",
)
automasker = AutoMasker(
    densepose_ckpt=os.path.join(ruta_modelo, "DensePose"),
    schp_ckpt=os.path.join(ruta_modelo, "SCHP"),
    device="cuda",
)
procesador_mascara = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)
print("Modelo listo")

## 5. Preparar los uniformes y la función que viste a la persona

- A los uniformes se les quita el fondo negro y quedan sobre blanco, como las fotos con las que aprendió el modelo.
- La foto de la persona no se recorta: se rellena por los bordes hasta la forma 3:4 que pide el modelo.
- Al final solo se cambia la zona de la ropa y los zapatos; la cara, las manos y el fondo quedan con los píxeles originales.
- CatVTON protege los zapatos a propósito, por eso se agregan aparte a la zona que se repinta.

In [ ]:
#@title 5. Preparar los uniformes y la función que viste a la persona
from rembg import remove

ANCHO, ALTO = 768, 1024
ZAPATOS_LIP = [18, 19]
ZAPATOS_ATR = [9, 10]


def preparar_uniforme(imagen):
    recorte = remove(imagen.convert("RGB"))
    recorte = recorte.crop(recorte.getbbox())
    fondo = Image.new("RGB", recorte.size, (255, 255, 255))
    fondo.paste(recorte, mask=recorte.split()[3])
    return fondo


def rellenar_a_3x4(imagen):
    # Devuelve la imagen rellenada y la caja donde quedó la foto original
    w, h = imagen.size
    if w * ALTO > h * ANCHO:
        nw, nh = w, round(w * ALTO / ANCHO)
    else:
        nw, nh = round(h * ANCHO / ALTO), h
    izq, arriba = (nw - w) // 2, (nh - h) // 2
    arr = cv2.copyMakeBorder(np.array(imagen), arriba, nh - h - arriba, izq, nw - w - izq, cv2.BORDER_REPLICATE)
    return Image.fromarray(arr), (izq, arriba, izq + w, arriba + h)


def zona_zapatos(datos, tamano):
    lip = np.array(datos["schp_lip"].resize(tamano, Image.NEAREST))
    atr = np.array(datos["schp_atr"].resize(tamano, Image.NEAREST))
    zona = (np.isin(lip, ZAPATOS_LIP) | np.isin(atr, ZAPATOS_ATR)).astype(np.uint8) * 255
    return cv2.dilate(zona, np.ones((15, 15), np.uint8), iterations=1)


def vestir(persona, uniforme, pasos=50, guia=2.5, semilla=42):
    persona = ImageOps.exif_transpose(persona).convert("RGB")
    lienzo, caja = rellenar_a_3x4(persona)
    lienzo_modelo = lienzo.resize((ANCHO, ALTO), Image.LANCZOS)

    datos = automasker(lienzo_modelo, "overall")
    ropa = np.array(datos["mask"].convert("L"))
    mascara = Image.fromarray(np.maximum(ropa, zona_zapatos(datos, (ANCHO, ALTO))))
    mascara = procesador_mascara.blur(mascara, blur_factor=9)

    generador = torch.Generator(device="cuda").manual_seed(int(semilla))
    resultado = pipeline(
        image=lienzo_modelo,
        condition_image=uniforme,
        mask=mascara,
        num_inference_steps=int(pasos),
        guidance_scale=guia,
        height=ALTO,
        width=ANCHO,
        generator=generador,
    )[0]

    # Volver al tamaño de la foto original y pegar solo la ropa nueva
    nitido = ImageFilter.UnsharpMask(radius=2, percent=80, threshold=2)
    resultado = resultado.resize(lienzo.size, Image.LANCZOS).crop(caja).filter(nitido)
    zona = mascara.convert("L").resize(lienzo.size, Image.BILINEAR).crop(caja)
    zona = zona.filter(ImageFilter.GaussianBlur(max(1, persona.height // 300)))
    return Image.composite(resultado, persona, zona)


# Si no se subió el ZIP (paso 3), los uniformes se bajan de la página de Render
import os, urllib.request
os.makedirs("/content/datos", exist_ok=True)
for nombre in ("uniforme_hombre.png", "uniforme_mujer.png"):
    ruta = f"/content/datos/{nombre}"
    if not os.path.exists(ruta):
        print("Bajando", nombre, "de la página (si estaba dormida tarda hasta 1 minuto)…")
        with urllib.request.urlopen(f"https://ropa-senati.onrender.com/estatico/{nombre}", timeout=180) as r:
            open(ruta, "wb").write(r.read())

uniformes = {
    "hombre": preparar_uniforme(Image.open("/content/datos/uniforme_hombre.png")),
    "mujer": preparar_uniforme(Image.open("/content/datos/uniforme_mujer.png")),
}

import matplotlib.pyplot as plt
fig, ejes = plt.subplots(1, 2, figsize=(8, 6))
for eje, (nombre, imagen) in zip(ejes, uniformes.items()):
    eje.imshow(imagen); eje.set_title(f"Uniforme {nombre}"); eje.axis("off")
plt.show()

## 6. Cambiar todas las fotos

Unos 40–60 segundos por foto en la T4. Las fotos de `hombre/` reciben el uniforme de hombre y las de `mujeres/` el de mujer.
Al terminar se descarga `resultados_uniforme.zip` con:
- `resultado/`: la foto final
- `antes_y_despues/`: la foto original al lado de la final

In [ ]:
#@title 6. Cambiar todas las fotos
import glob, shutil, time
from google.colab import files

PASOS = 50     # más pasos = más detalle, pero más lento (30 a 50)
SEMILLA = 42   # cambia este número si una foto sale rara

salida = "/content/salida"
shutil.rmtree(salida, ignore_errors=True)
os.makedirs(f"{salida}/resultado"); os.makedirs(f"{salida}/antes_y_despues")

carpetas = {"hombre": "hombre", "mujeres": "mujer"}
listos, fallidos = [], []
inicio = time.time()
for carpeta, tipo in carpetas.items():
    for ruta in sorted(glob.glob(f"/content/datos/{carpeta}/*")):
        nombre = os.path.splitext(os.path.basename(ruta))[0]
        try:
            original = ImageOps.exif_transpose(Image.open(ruta)).convert("RGB")
            final = vestir(original, uniformes[tipo], PASOS, semilla=SEMILLA)
            final.save(f"{salida}/resultado/{nombre}.png")
            lado = Image.new("RGB", (original.width * 2 + 10, original.height), (255, 255, 255))
            lado.paste(original, (0, 0)); lado.paste(final, (original.width + 10, 0))
            lado.save(f"{salida}/antes_y_despues/{nombre}.jpg", quality=92)
            listos.append(f"{salida}/antes_y_despues/{nombre}.jpg")
            print(f"✔ {nombre}  ({time.time() - inicio:.0f} s)")
        except Exception as error:
            fallidos.append(nombre)
            print(f"✘ {nombre}: {error}")

print(f"\nListas: {len(listos)}  Fallidas: {len(fallidos)} {fallidos}")
shutil.make_archive("/content/resultados_uniforme", "zip", salida)
files.download("/content/resultados_uniforme.zip")

# Vista previa de las primeras 6
fig, ejes = plt.subplots(2, 3, figsize=(15, 10))
for eje, ruta in zip(ejes.flat, listos[:6]):
    eje.imshow(Image.open(ruta)); eje.set_title(os.path.basename(ruta)); eje.axis("off")
for eje in list(ejes.flat)[len(listos[:6]):]:
    eje.axis("off")
plt.show()

## 7. Página para subir fotos sueltas

Abre el enlace `https://....gradio.live` que aparece abajo (funciona también en el celular mientras el cuaderno siga abierto).
Para cerrarla, detén esta celda con ■.

In [ ]:
#@title 7. Página para subir fotos sueltas
import gradio as gr


def desde_pagina(foto, tipo, otro_uniforme, pasos, semilla):
    if foto is None:
        raise gr.Error("Primero sube una foto de la persona.")
    if otro_uniforme is not None:
        uniforme = preparar_uniforme(otro_uniforme)
    else:
        uniforme = uniformes["hombre" if tipo == "Chico (uniforme de hombre)" else "mujer"]
    return vestir(foto, uniforme, pasos, semilla=semilla)


with gr.Blocks(title="Uniforme SENATI") as pagina:
    gr.Markdown("## Ponerle el uniforme SENATI a una foto\nMisma cara, misma pose y mismo fondo; solo cambia la ropa.")
    with gr.Row():
        with gr.Column():
            foto = gr.Image(type="pil", label="Foto de la persona (OBLIGATORIO)")
            tipo = gr.Radio(["Chico (uniforme de hombre)", "Chica (uniforme de mujer)"],
                            value="Chica (uniforme de mujer)", label="Uniforme")
            otro = gr.Image(type="pil", label="Otro uniforme (OPCIONAL, reemplaza al de arriba)")
            with gr.Accordion("Ajustes", open=False):
                pasos = gr.Slider(20, 60, value=50, step=5, label="Pasos (mín. 20, máx. 60)")
                semilla = gr.Number(value=42, precision=0, label="Semilla (cámbiala si sale rara)")
            boton = gr.Button("Poner uniforme", variant="primary")
        resultado = gr.Image(type="pil", label="Resultado", format="png")
    boton.click(desde_pagina, [foto, tipo, otro, pasos, semilla], resultado)

pagina.queue().launch(share=True, debug=True)

## 8. Motor del evento (conecta con la página de Render)

La clave del motor ya va escrita aquí: **no compartas este cuaderno**.
Deja esta pestaña abierta todo el evento. Para apagar el motor, detén la celda con ■.

In [ ]:
#@title 8. Motor del evento (conecta con la página de Render)
import sys, subprocess
subprocess.run(["wget", "-q", "-O", "/content/motor.py",
                "https://raw.githubusercontent.com/quintillizasyuesugui-svg/ropa-senati/main/motor.py"], check=True)
sys.path.insert(0, "/content")
import importlib, motor
importlib.reload(motor)
clave = "PEGA-AQUI-LA-CLAVE-DEL-MOTOR"
motor.encender(pipeline, vestir, uniformes, clave)